In [ ]:
import os

# Konfiguracja API Kaggle
os.environ['KAGGLE_USERNAME'] = "sikoraa36" 
os.environ['KAGGLE_KEY'] = "KLUCZ_API" 
# !kaggle datasets download -d krzysztofjamroz/apartment-prices-in-poland --unzip

In [ ]:
import pandas as pd
import glob

# 1. Szukamy wszystkich plików ze sprzedażą (zaczynają się od "apartments_pl_") Ignorujemy pliki "rent" (wynajem)
files = sorted(glob.glob("apartments_pl_*.csv"))

print(f"Znaleziono {len(files)} plików do połączenia.")

# 2. Wczytujemy je po kolei i dodajemy do listy
dfs = []
for filename in files:
    temp_df = pd.read_csv(filename)
    # Wyciągamy rok i miesiąc z nazwy pliku, żeby wiedzieć kiedy była oferta
    # np. z 'apartments_pl_2023_09.csv' robimy '2023_09'
    month_id = filename.replace("apartments_pl_", "").replace(".csv", "")
    temp_df['year_month'] = month_id 
    dfs.append(temp_df)

# 3. Sklejamy w jeden wielki DataFrame
df = pd.concat(dfs, ignore_index=True)

print(f"Łączna liczba ofert w Twojej bazie: {df.shape[0]}")

# Tworzymy kolumnę price_per_m2 (Cena całkowita / Metraż)
df['price_per_m2'] = df['price'] / df['squareMeters']

display(df.sample(5))

In [ ]:
# Sprawdzamy procent brakujących danych w każdej kolumnie
missing_data = df.isnull().mean() * 100
# tylko te kolumny, gdzie faktycznie czegoś brakuje (więcej niż 0%)
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)
print("Procent brakujących danych w kolumnach: ")
print(missing_data)

In [ ]:
def clean_data(df):
    df_clean = df.copy()

    # USUNIĘCIE DUPLIKATÓW
    df_clean = df_clean.drop_duplicates()

    # Wypełniamy braki słowem 'unknown', żeby nie tracić kluczowych informacji
    cols_to_fix = ['condition', 'buildingMaterial', 'type']
    for col in cols_to_fix:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna('unknown')

    # UZUPEŁNIANIE DANYCH LICZBOWYCH
    df_clean['hasElevator'] = df_clean['hasElevator'].replace({"yes": 1, "no": 0})
    # Dla kolumn z odległościami od centrum miasta wypełniamy medianą
    distance_cols = [col for col in df_clean.columns if 'Distance' in col]
    for col in distance_cols:
        median_value = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(median_value)

    # FILTROWANIE OUTLIERÓW (Ceny i metraże z kosmosu)
    # Przyjmujemy rozsądne ramy: np. cena za metr między 2k a 50k, metraż 15-300m2
    df_clean = df_clean[ (df_clean['price_per_m2'] > 2000) & (df_clean['price_per_m2'] < 60000) ]
    df_clean = df_clean[ (df_clean['squareMeters'] > 15) & (df_clean['squareMeters'] < 300) ]
    
    # CZYSZCZENIE BRAKÓW W KLUCZOWYCH KOLUMNACH
    critical_cols = ['floor', 'buildYear', 'floorCount']
    df_clean = df_clean.dropna(subset=critical_cols)
    
    return df_clean

df = clean_data(df)

print("\nSprawdzenie końcowe:")
print(df.isnull().sum().sum()) # Jeśli wynik to 0, to jest idealnie!

In [ ]:
display(df.describe())

In [ ]:
# Filtrujemy tylko bardzo stare budynki (sprzed 1900 roku)
stare_kamienice = df[df['buildYear'] < 1900]

# Sprawdźmy, w jakich miastach występują najczęściej
print("\n--- Gdzie są te stare budynki? (Top 5 miast) ")
print(stare_kamienice['city'].value_counts().head(5))

# Zobaczmy przykładowe oferty (Cena za m2 pokaże nam, czy to ruina czy luksus)
display(stare_kamienice[['city', 'buildYear', 'price', 'squareMeters']].sort_values(by='buildYear', ascending=True).head(20))


In [ ]:
# Sprawdzamy mieszkania powyżej 20. piętra
wysokie_pietra = df[df['floor'] > 20]

# W jakich miastach się znajdują
print("\nGdzie są wieżowce?")
print(wysokie_pietra['city'].value_counts())

# Spójrzmy na ceny - powinny być BARDZO wysokie (to luksusowe apartamenty)
display(wysokie_pietra[['city', 'floor', 'price_per_m2', "squareMeters"]].sort_values(by='floor', ascending=False).head(15))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Wybieramy tylko kolumny liczbowe
df_numeric = df.select_dtypes(include=['float64', 'int64'])

# Obliczamy korelację
correlation_matrix = df_numeric.corr()

# Heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title("Macierz Korelacji")
plt.show()

print("Co najbardziej podbija cenę za m2?")
# Sortujemy korelacje względem ceny za m2
print(correlation_matrix['price_per_m2'].sort_values(ascending=False))

In [ ]:
# Wybieramy tylko te kolumny, które chcemy dać modelowi
# Odrzucamy te mało istotne (id) oraz te, które zdradzają odpowiedź (price - bo przewidujemy cenę)
features = [
    'city', 'squareMeters', 'rooms', 'floor', 'floorCount', 'buildYear',
    'latitude', 'longitude', 'centreDistance', 'poiCount', 'schoolDistance',
    'clinicDistance', 'pharmacyDistance', 'kindergartenDistance', 'restaurantDistance', 'condition', 'buildingMaterial',
    'hasElevator' # hasElevator to 0 lub 1, więc jest ok
]

X = df[features]
y = df['price'] # Target

# Zamiana miast na bool (One-Hot Encoding)
# Zmiana 'city' na 'city_Warszawa', 'city_Wroclaw' itd.
X = pd.get_dummies(X, columns=['city', 'condition', 'buildingMaterial'], dtype=int)

print("Gotowe dane do modelu")
print(f"Liczba kolumn po transformacji: {X.shape[1]}")
display(X.head())

In [ ]:
from sklearn.model_selection import train_test_split

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Rozmiar zbioru treningowego: {X_train.shape}")
print(f"Rozmiar zbioru testowego: {X_test.shape}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Inicjalizacja modelu
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Średni błąd (MAE): {mae:.2f} PLN")
print(f"R2 Score: {r2:.4f}")

In [ ]:
# Feature Importance
importances = model.feature_importances_
feature_names = X.columns
forest_importances = pd.Series(importances, index=feature_names).sort_values(ascending=False).head(20)

# Wykres
plt.figure(figsize=(12, 8))
forest_importances.plot.bar()
plt.title("Co najbardziej wpływa na cenę mieszkania? (Top 20 cech)")
plt.ylabel("Ważność cechy")
plt.show()

In [ ]:
import shap

# Wyjaśnienie modelu
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test.iloc[:100]) # sample 100 mieszkań dla szybkości

# Wykres, który pokazuje co wpływa na cenę
shap.summary_plot(shap_values, X_test.iloc[:100])

In [ ]:
# WYKRYWANIE OKAZJI
all_predictions = model.predict(X)

analysis_df = df.copy()
analysis_df['predicted_price'] = all_predictions
analysis_df['difference_pln'] = analysis_df['predicted_price'] - analysis_df['price']
analysis_df['opportunity_score'] = analysis_df['difference_pln'] / analysis_df['predicted_price']

# Sortujemy po % różnicy (model wycenił na 100%, cena to 70% -> mieszkanie jest niedowartościowane)
deals = analysis_df.sort_values(by='opportunity_score', ascending=False)

print("TOP 5 OKAZJI NA RYNKU:")
cols_show = ['city', 'squareMeters', 'price', 'predicted_price', 'difference_pln']
display(deals[cols_show].head(5))

# Sprawdzenie czy okazje to ruiny
plt.figure(figsize=(10, 6))
sns.scatterplot(data=analysis_df, x='price', y='predicted_price', alpha=0.3)
plt.plot([0, 3500000], [0, 3500000], 'r--', linewidth=2) # Linia idealna
plt.ticklabel_format(style='plain')
plt.title("Cena Rzeczywista vs Przewidywana (Punkty nad linią to okazje)")
plt.xlabel("Cena z ogłoszenia")
plt.ylabel("Wycena Modelu")
plt.show()